# Sentence Transformers

Sentence Transformers provides pretrained models that map text into dense numerical vectors called **embeddings**.

Texts with similar meanings tend to have vectors that are close to one another in the embedding space.

This notebook uses pretrained models from the `sentence-transformers` library. Internet access may be required the first time a model is downloaded.

## 1. Import the libraries

In [1]:
# Import the required libraries
# In case you require to install these packages for the first time:
# %pip install -U sentence-transformers scikit-learn pandas numpy
import numpy as np
import pandas as pd
# KMeans for grouping text embeddings into clusters.
from sklearn.cluster import KMeans
# cosine similarity for comparing embedding vectors.
from sklearn.metrics.pairwise import cosine_similarity

# Import evaluation and classification utilities.
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Import normalization for preparing embedding features.
from sklearn.preprocessing import normalize

# Loading pretrained embedding models.
from sentence_transformers import SentenceTransformer

# Load a compact and effective English sentence-embedding model.
model = SentenceTransformer("all-MiniLM-L6-v2")

print("Model loaded successfully.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded successfully.


## 2. Convert Text into Vectors

A Sentence Transformer converts a text input into a fixed-length dense vector.

For example:

- Input: `"Machine learning is useful."`
- Output: A numerical vector such as `[0.12, -0.08, ..., 0.31]`

The actual vector contains many dimensions and is learned by the pretrained model.

Sentence Transformers convert texts of any lengths
into a fixed-length dense numerical vector, known as embedding. This fixed length is a characteristic
of the specific embedding model being used.

This consistency allows for direct comparison
and mathematical operations between embeddings,
which is crucial for tasks like calculating cosine similarity.


In [2]:
# 1. Add two more sentences to texts.

texts = [
    "Machine learning is useful.",
    "Artificial intelligence can solve complex problems.",
    "The weather is sunny today.",
    "Deep Learning is the subset of Machine Learning.",
    "I decided to choose this topic for our group."
]

# 2. Encode the updated list.
embeddings = model.encode(texts)

# 3. Determine the number of dimensions in each embedding.
print("Embedding matrix shape:", embeddings.shape)
print("First embedding preview:", embeddings[0][:10])

Embedding matrix shape: (5, 384)
First embedding preview: [ 0.01712887 -0.04781876  0.0411445  -0.00472872 -0.00806266  0.04857659
 -0.05506266 -0.01392579 -0.02275805 -0.00896066]


## 3. Text Similarity with Cosine Similarity

**Cosine similarity** measures the angle between two vectors.

- A value close to `1` indicates strong similarity.
- A value close to `0` indicates weak or unrelated similarity.
- A negative value may indicate opposing directions, although the interpretation depends on the model.

The formula is:

$$
\operatorname{cosine\_similarity}(u,v)
=
\frac{u \cdot v}{\|u\|\|v\|}
$$


In [3]:
def text_similarity(sentence_pairs):
  # Encode every sentence in the pairs.
  pair_embeddings = model.encode(
        [sentence for pair in sentence_pairs for sentence in pair]
  )

  # Compare the two sentences in each pair.
  for index, (sentence_a, sentence_b) in enumerate(sentence_pairs):
      # Extract the two embeddings belonging to the current pair.
      # Since `pair_embeddings` is a flattened list of all sentence embeddings
      # (two per pair), we use `index * 2` and `index * 2 + 1` to get
      # the embeddings for `sentence_a` and `sentence_b` of the current pair.
      vector_a = pair_embeddings[index * 2]
      vector_b = pair_embeddings[index * 2 + 1]

      # Calculate cosine similarity between the two vectors.
      score = cosine_similarity(
          vector_a.reshape(1, -1),
          vector_b.reshape(1, -1)
      )[0][0]

      # Display the pair and its similarity score.
      print(f"Pair {index + 1}")
      print(f"Sentence A: {sentence_a}")
      print(f"Sentence B: {sentence_b}")
      print(f"Cosine similarity: {score:.4f}")
      print("-" * 60)

In [4]:
sentence_pairs = [
    ("A dog is running in the park.", "A puppy is playing outdoors."),
    ("A dog is running in the park.", "The stock market closed higher today."),
    ("How can I reset my password?", "What should I do if I forget my password?"),
    ("There is a man running.", "There is a man jogging"),
    ("The moon is blue.", "The man is playing video game at night."),
    ("What is your home address?", "Can I please have your email address?")
]

text_similarity(sentence_pairs)

Pair 1
Sentence A: A dog is running in the park.
Sentence B: A puppy is playing outdoors.
Cosine similarity: 0.4453
------------------------------------------------------------
Pair 2
Sentence A: A dog is running in the park.
Sentence B: The stock market closed higher today.
Cosine similarity: 0.0319
------------------------------------------------------------
Pair 3
Sentence A: How can I reset my password?
Sentence B: What should I do if I forget my password?
Cosine similarity: 0.8105
------------------------------------------------------------
Pair 4
Sentence A: There is a man running.
Sentence B: There is a man jogging
Cosine similarity: 0.7884
------------------------------------------------------------
Pair 5
Sentence A: The moon is blue.
Sentence B: The man is playing video game at night.
Cosine similarity: 0.0308
------------------------------------------------------------
Pair 6
Sentence A: What is your home address?
Sentence B: Can I please have your email address?
Cosine simi

## 4. Semantic Search

Traditional keyword search mainly looks for exact or related words.

**Semantic search** represents both the query and documents as embeddings. It then ranks documents according to vector similarity. This allows the system to retrieve relevant documents even when the query and document use different words.


In [5]:
def semantic_search(query, documents):
  # Encode the documents and the query.
  document_embeddings = model.encode(documents)
  query_embedding = model.encode([query])

  # Calculate the similarity between the query and every document.
  similarity_scores = cosine_similarity(
      query_embedding,
      document_embeddings
  )[0]

  # Rank document indexes from the highest score to the lowest score.
  ranked_indexes = np.argsort(similarity_scores)[::-1]

  # Display the ranked search results.
  for rank, document_index in enumerate(ranked_indexes, start=1):
      print(f"{rank}. Score: {similarity_scores[document_index]:.4f}")
      print(f"   {documents[document_index]}")

In [6]:
# Define a natural-language search query.
documents = [
    "Python is a popular programming language for data science.",
    "Deep learning uses neural networks with multiple layers.",
    "A database stores and organizes structured information.",
    "Computer networks allow devices to communicate with one another.",
    "Natural language processing enables computers to work with human language."
]

In [7]:
query1 = "How do computers communicate?"
print("Answer to Query 1:")
semantic_search(query1, documents)

query2 = "What is used to analyze large datasets?"
print("\nAnswer to Query 2:")
semantic_search(query2, documents)

query3 = "How do neural networks learn complex patterns?"
print("\nAnswer to Query 3:")
semantic_search(query3, documents)

Answer to Query 1:
1. Score: 0.6769
   Computer networks allow devices to communicate with one another.
2. Score: 0.4044
   Natural language processing enables computers to work with human language.
3. Score: 0.2588
   Deep learning uses neural networks with multiple layers.
4. Score: 0.2390
   Python is a popular programming language for data science.
5. Score: 0.2093
   A database stores and organizes structured information.

Answer to Query 2:
1. Score: 0.3850
   A database stores and organizes structured information.
2. Score: 0.3342
   Python is a popular programming language for data science.
3. Score: 0.2951
   Deep learning uses neural networks with multiple layers.
4. Score: 0.2317
   Computer networks allow devices to communicate with one another.
5. Score: 0.2092
   Natural language processing enables computers to work with human language.

Answer to Query 3:
1. Score: 0.4959
   Deep learning uses neural networks with multiple layers.
2. Score: 0.2583
   Computer networks al

## 5. Retrieval-Based FAQ Chatbot

A retrieval-based chatbot does not generate a new answer. Instead, it:

1. Encodes a user's question.
2. Finds the most similar FAQ question.
3. Returns the answer associated with that FAQ.
4. Optionally rejects the query if the similarity score is too low.

This approach is simple, explainable, and useful for narrow-domain support systems.


***The Role of the Threshold***

The `threshold` parameter in a retrieval-based chatbot is crucial for controlling the relevance of the answers provided. It acts as a minimum similarity score that a matched FAQ question must achieve with the user's query for an answer to be returned.

-   **If the similarity score is above the threshold**: The chatbot considers the matched FAQ sufficiently relevant and returns its associated answer.
-   **If the similarity score is below the threshold**: The chatbot determines that no existing FAQ is a good enough match for the user's query. In this case, it typically returns a default message indicating that it could not find a relevant answer, effectively rejecting the question.

Setting an appropriate threshold helps to:

-   **Prevent irrelevant answers**: It avoids providing answers that are only weakly related to the user's question, which could lead to user frustration or incorrect information.
-   **Improve user experience**: Users get a clearer indication when the chatbot cannot help, rather than a potentially misleading answer.
-   **Control chatbot behavior**: Adjusting the threshold allows fine-tuning the chatbot's strictness. A higher threshold means the chatbot is more selective, only answering very similar questions, while a lower threshold makes it more permissive, potentially returning answers for questions that are less directly related.

In [8]:
def answer_faq(user_question, threshold=0.45):
    # Encode the incoming user question.
    question_embedding = model.encode([user_question])

    # Calculate similarity between the user question and every FAQ question.
    scores = cosine_similarity(
        question_embedding,
        faq_embeddings
    )[0]

    # Find the index of the most similar FAQ question.
    best_index = int(np.argmax(scores))

    # Retrieve the highest similarity score.
    best_score = float(scores[best_index])

    # Reject the question when no FAQ is sufficiently similar.
    if best_score < threshold:
        return {
            "answer": "Sorry, I could not find a relevant FAQ answer.",
            "matched_question": None,
            "score": best_score
        }

    # Return the answer associated with the best matching FAQ.
    return {
        "answer": faq_data[best_index]["answer"],
        "matched_question": faq_data[best_index]["question"],
        "score": best_score
    }

In [9]:
# 1. Add at least three new FAQs.
faq_data = [
    {
        "question": "How do I reset my password?",
        "answer": "Open the login page, select 'Forgot password', and follow the instructions."
    },
    {
        "question": "How can I update my email address?",
        "answer": "Go to Account Settings, edit your email address, and save the changes."
    },
    {
        "question": "Where can I view my payment history?",
        "answer": "Open Billing, then select Payment History."
    },
    {
        "question": "How do I cancel my subscription?",
        "answer": "Go to Subscription Settings and select Cancel Subscription."
    },

    {
        "question": "I need help using the chatbot functionality?",
        "answer": "Visit the chat section at the top left to use the chatbot."
    },

    {
        "question": "Where do I submit my completed report?",
        "answer": "Open the Report section and find the Submit button to upload your report."
    },

    {
        "question": "Can I schedule a new meeting through the system?",
        "answer": "Find the Create button at the top right and scroll down to find 'New Meeting' option."
    }
]

# Extract FAQ questions from the knowledge base.
faq_questions = [item["question"] for item in faq_data]

# Encode all FAQ questions once so they can be reused.
faq_embeddings = model.encode(faq_questions)

In [10]:
# Test questions that use different wording from the stored FAQ questions.
result = answer_faq("I need to submit this report. What should I do?")

print("Matched FAQ:", result["matched_question"])
print("Similarity score:", f"{result['score']:.4f}")
print("Chatbot answer:", result["answer"])

# Test an unrelated question
result = answer_faq("I have a question but I don't know who to ask?")

print("Matched FAQ:", result["matched_question"])
print("Similarity score:", f"{result['score']:.4f}")
print("Chatbot answer:", result["answer"])

Matched FAQ: Where do I submit my completed report?
Similarity score: 0.7475
Chatbot answer: Open the Report section and find the Submit button to upload your report.
Matched FAQ: None
Similarity score: 0.0538
Chatbot answer: Sorry, I could not find a relevant FAQ answer.


In [11]:
# Adjust the similarity threshold and observe how the chatbot behavior changes.
user_question = "I need to submit this report. What should I do?"

result_thresholded = answer_faq(user_question, threshold=0.05)


print("Matched FAQ:", result_thresholded["matched_question"])
print("Similarity score:", f"{result_thresholded['score']:.4f}")
print("Chatbot answer:", result_thresholded["answer"])

Matched FAQ: Where do I submit my completed report?
Similarity score: 0.7475
Chatbot answer: Open the Report section and find the Submit button to upload your report.


## 6. Text Clustering

Clustering groups texts without requiring predefined labels.

A common workflow is:

1. Encode documents into embeddings.
2. Select a clustering algorithm.
3. Group similar vectors.
4. Inspect the documents in each cluster to identify the topics.

Here, K-Means is used for demonstration.


Clustering labels, such as 0, 1, or 2, are arbitrary identifiers
assigned by the clustering algorithm to group similar data points
(in this case, text embeddings).

They simply denote a group, but do not carry
any intrinsic meaning or represent a specific topic name.

The interpretation of what each cluster represents (i.e., its 'topic')
comes from a human analyst inspecting the documents
within each cluster and inferring common themes.

For example, if cluster 0 contains mostly news about sports,
then a human can label it 'Sports News',
but the algorithm itself only sees it as 'cluster 0'.

In [12]:
def text_clustering(news_documents):
  # Convert all news documents into embedding vectors.
  news_embeddings = model.encode(news_documents)

  # Create a K-Means model with three clusters.
  kmeans = KMeans(
      n_clusters=4,
      random_state=42,
      n_init=10
  )

  # Assign each document to a cluster.
  cluster_labels = kmeans.fit_predict(news_embeddings)

  # Display the documents grouped by their predicted cluster.
  clustered_news = pd.DataFrame({
      "document": news_documents,
      "cluster": cluster_labels
  })

  print(clustered_news.sort_values("cluster").to_string(index=False))

In [13]:
news_documents = [
    "The football team won the championship after a close match.",
    "The player scored two goals in the final game.",
    "The government announced a new economic policy.",
    "Interest rates may affect the national economy.",
    "A new smartphone includes an advanced camera system.",
    "Technology companies are developing faster processors.",
    "The basketball season begins next week.",
    "The central bank discussed inflation and growth.",
    "The latest laptop uses an energy-efficient chip.",
    "A girl who was missing three days ago have been found alive.",
    "The suspect is finally convicted of their crimes.",
    "It is still unknown why she was never reported missing by her parents."
]

text_clustering(news_documents)

                                                              document  cluster
           The football team won the championship after a close match.        0
                        The player scored two goals in the final game.        0
                               The basketball season begins next week.        0
                     The suspect is finally convicted of their crimes.        0
                       Interest rates may affect the national economy.        1
                      The central bank discussed inflation and growth.        1
                       The government announced a new economic policy.        1
                Technology companies are developing faster processors.        2
                  A new smartphone includes an advanced camera system.        2
                      The latest laptop uses an energy-efficient chip.        2
          A girl who was missing three days ago have been found alive.        3
It is still unknown why she was never re

## 7. Text Classification Using Embeddings

Embeddings can be used as input features for a traditional machine-learning classifier.

Workflow:

1. Prepare labeled text examples.
2. Convert the texts into embeddings.
3. Split the data into training and testing sets.
4. Train a classifier such as Logistic Regression.
5. Evaluate its predictions.

The embedding model acts as a general-purpose feature extractor.


***Explanation of differences:***

- **The embedding model:** This is a Sentence Transformer model (like 'all-MiniLM-L6-v2')
that converts raw text into fixed-length numerical vectors (embeddings).

Its role is to capture the semantic meaning of text in a dense, numerical format.
It does not perform classification directly; it merely provides a numerical representation of the text.

- **The classification model:** This is a traditional machine learning model (like Logistic Regression in this case)
that takes the embeddings generated by the embedding model as its input features.

Its role is to learn a mapping from these numerical features
to predefined categories or labels (e.g., 'account', 'billing', 'technical', 'shipping')
based on the training data.

It is responsible for making the decision of which category a given embedding belongs to.

- **The final predicted label:** This is the output of the classification model for a given input text's embedding.
It is the specific category or class (e.g., 'account') that the classifier assigns to the text,
representing the model's best guess about the intent or topic of the input text.

In [14]:
def text_classification(classification_data):
  # Separate the text examples from their labels.
  classification_texts = [item[0] for item in classification_data]
  classification_labels = [item[1] for item in classification_data]

  # Convert the text examples into embedding features.
  classification_embeddings = model.encode(classification_texts)

  # Split the dataset into training and testing portions.
  X_train, X_test, y_train, y_test = train_test_split(
      classification_embeddings,
      classification_labels,
      test_size=0.30,
      random_state=42,
      stratify=classification_labels
  )

  # Create a Logistic Regression classifier.
  classifier = LogisticRegression(
      max_iter=1000,
      random_state=42
  )

  # Train the classifier using the embedding features.
  classifier.fit(X_train, y_train)

  # Predict labels for the test examples.
  y_pred = classifier.predict(X_test)

  # Evaluate the classifier.
  print("Accuracy:", f"{accuracy_score(y_test, y_pred):.4f}")
  print(classification_report(y_test, y_pred, zero_division=0))

In [15]:
classification_data = [
    ("I forgot my password", "account"),
    ("How can I change my username?", "account"),
    ("I want to update my profile", "account"),
    ("My login isn't working", "account"),
    ("Can I recover my account?", "account"),

    ("My payment was declined", "billing"),
    ("Where is my invoice?", "billing"),
    ("How do I update my credit card?", "billing"),
    ("Why was I charged twice?", "billing"),
    ("I need a refund", "billing"),

    ("The application crashes when I open it", "technical"),
    ("The website shows an error message", "technical"),
    ("The software is not working correctly", "technical"),
    ("The mobile app freezes frequently", "technical"),
    ("I can't install the update", "technical"),

    ("Where is my order?", "shipping"),
    ("My package is delayed", "shipping"),
    ("How can I track my delivery?", "shipping"),
    ("I received the wrong item", "shipping"),
    ("Can I change my delivery address?", "shipping")
]

text_classification(classification_data)

Accuracy: 0.8333
              precision    recall  f1-score   support

     account       1.00      1.00      1.00         2
     billing       0.50      1.00      0.67         1
    shipping       1.00      0.50      0.67         2
   technical       1.00      1.00      1.00         1

    accuracy                           0.83         6
   macro avg       0.88      0.88      0.83         6
weighted avg       0.92      0.83      0.83         6



## 8. Recommendation System

An embedding-based recommendation system represents item descriptions as vectors.

For example, it can recommend:

- Products similar to a product description.
- Courses related to a student's interests.
- Articles related to a selected article.

The recommendation score is commonly computed with cosine similarity.


In [23]:
def recommend_system(documents, descriptions, user_query, title, content):
  # Encode the user's query.
  query_embedding = model.encode([user_query])
  # Encode all descriptions.
  descriptions_embeddings = model.encode(descriptions)

  # Calculate similarity between the query and every technical document.
  scores = cosine_similarity(
      query_embedding,
      descriptions_embeddings
  )[0]

  # Rank documents by similarity score.
  ranking = np.argsort(scores)[::-1]

  # Display recommended technical documents.
  print(f"Recommendations for: '{user_query}'\n")
  for rank, doc_index in enumerate(ranking, start=1):
      print(f"{rank}. {documents[doc_index][title]}")
      print(f"   Score: {scores[doc_index]:.4f}")
      print(f"   {documents[doc_index][content]}")
      print("-" * 60)

In [25]:
courses = [
    {
        "title": "Introduction to Python",
        "description": "Learn Python programming, variables, loops, functions, and data structures."
    },
    {
        "title": "Data Science Fundamentals",
        "description": "Study data analysis, visualization, statistics, and practical data science workflows."
    },
    {
        "title": "Deep Learning with Neural Networks",
        "description": "Build neural network models for image, text, and prediction tasks."
    },
    {
        "title": "Database Systems",
        "description": "Learn relational databases, SQL queries, normalization, and database design."
    },
    {
        "title": "Natural Language Processing",
        "description": "Process human language using tokenization, embeddings, transformers, and text analysis."
    }
]

courses_discriptions = [course["description"] for course in courses]
learner_interest = "I want to learn how to analyze datasets and create useful visualizations."
recommend_system(courses, courses_discriptions, learner_interest,
                 title='title', content='description')

Recommendations for: 'I want to learn how to analyze datasets and create useful visualizations.'

1. Data Science Fundamentals
   Score: 0.6210
   Study data analysis, visualization, statistics, and practical data science workflows.
------------------------------------------------------------
2. Database Systems
   Score: 0.4320
   Learn relational databases, SQL queries, normalization, and database design.
------------------------------------------------------------
3. Introduction to Python
   Score: 0.4289
   Learn Python programming, variables, loops, functions, and data structures.
------------------------------------------------------------
4. Deep Learning with Neural Networks
   Score: 0.1895
   Build neural network models for image, text, and prediction tasks.
------------------------------------------------------------
5. Natural Language Processing
   Score: 0.1384
   Process human language using tokenization, embeddings, transformers, and text analysis.
--------------------

*Example: Technical Documentation Recommendation System*

-   **Information to be embedded:**
The content of the technical documents themselves (e.g., titles, abstracts, full text sections, summaries).
Each document or relevant section should be converted into an embedding.

-   **Information to be used as the query:**
The user's specific technical question, problem description,
or keywords describing the information they are looking for.

In [26]:
# Define a collection of technical documentation snippets.
technical_documents = [
    {
        "title": "Configuring Network Settings",
        "content": "This document describes how to configure IP addresses, subnet masks, and default gateways on your device."
    },
    {
        "title": "Troubleshooting Common Software Errors",
        "content": "Learn to diagnose and resolve frequently encountered software issues like application crashes and installation failures."
    },
    {
        "title": "API Integration Guide",
        "content": "A comprehensive guide on integrating our API into your existing applications, covering authentication, data formats, and error handling."
    },
    {
        "title": "Hardware Maintenance Best Practices",
        "content": "Tips and procedures for maintaining hardware components to ensure longevity and optimal performance, including cleaning and upgrades."
    },
    {
        "title": "Security Protocol Implementation",
        "content": "Implementing secure communication protocols, encryption standards, and access control measures for data protection."
    }
]

# Extract document content for embedding.
technical_content = [doc["content"] for doc in technical_documents]
# Define a user's technical query.
user_technical_query = "I need help setting up the internet on my computer."

# Generate recommendations based on the user's query.
recommend_system(technical_documents, technical_content, user_technical_query,
                 title='title', content='content')

Recommendations for: 'I need help setting up the internet on my computer.'

1. Configuring Network Settings
   Score: 0.4346
   This document describes how to configure IP addresses, subnet masks, and default gateways on your device.
------------------------------------------------------------
2. Troubleshooting Common Software Errors
   Score: 0.1951
   Learn to diagnose and resolve frequently encountered software issues like application crashes and installation failures.
------------------------------------------------------------
3. Security Protocol Implementation
   Score: 0.1930
   Implementing secure communication protocols, encryption standards, and access control measures for data protection.
------------------------------------------------------------
4. API Integration Guide
   Score: 0.1715
   A comprehensive guide on integrating our API into your existing applications, covering authentication, data formats, and error handling.
----------------------------------------------

## 9. Paraphrase Detection

Two sentences are paraphrases when they express approximately the same meaning using different wording.

A simple paraphrase detector:

1. Encodes both sentences.
2. Calculates cosine similarity.
3. Compares the score with a threshold.

A threshold-based method is useful for demonstrations, but a production system should validate the threshold on labeled data.

***Adjusting the threshold***

In paraphrase detection, a false positive occurs when two sentences are incorrectly identified as paraphrases (e.g., threshold is too low). A false negative occurs when two actual paraphrases are missed (e.g., threshold is too high).

Adjusting the similarity threshold creates a trade-off: a lower threshold increases false positives but reduces false negatives, while a higher threshold reduces false positives but increases false negatives. The optimal threshold depends on the specific application's tolerance for each type of error.

In [27]:
def detect_paraphrase(sentence_a, sentence_b, threshold=0.70):
    # Encode both sentences as vectors.
    embeddings = model.encode([sentence_a, sentence_b])

    # Calculate cosine similarity between the two sentence vectors.
    score = cosine_similarity(
        embeddings[0].reshape(1, -1),
        embeddings[1].reshape(1, -1)
    )[0][0]

    # Decide whether the pair is likely to be a paraphrase.
    is_paraphrase = score >= threshold

    # Return both the score and the decision.
    return score, is_paraphrase

In [28]:
# Define sentence pairs for paraphrase detection.
sentence_pairs = [
    ("How do I change my password?", "What are the steps to reset my password?"),
    ("The cat is sleeping on the sofa.", "A cat is resting on the couch."),
    ("The meeting starts at nine.", "The meeting has been moved to eleven."),
    ("She bought a new laptop.", "She purchased a new notebook computer."),
    ("How do I change my profile picture?", "How does changining my profile picture work?"),
    ("The man is sleeping.", "The man is taking a nap."),
    ("The meeting has been cancelled.", "The meeting has been delayed to tonight."),
    ("She bought a new mobile phone.", "She purchased a new phone case.")
]

# Evaluate every sentence pair.
for sentence_a, sentence_b in sentence_pairs:
    score, decision = detect_paraphrase(sentence_a, sentence_b)

    print(f"Sentence A: {sentence_a}")
    print(f"Sentence B: {sentence_b}")
    print(f"Similarity: {score:.4f}")
    print(f"Likely paraphrase: {decision}")
    print("-" * 60)

Sentence A: How do I change my password?
Sentence B: What are the steps to reset my password?
Similarity: 0.8073
Likely paraphrase: True
------------------------------------------------------------
Sentence A: The cat is sleeping on the sofa.
Sentence B: A cat is resting on the couch.
Similarity: 0.7295
Likely paraphrase: True
------------------------------------------------------------
Sentence A: The meeting starts at nine.
Sentence B: The meeting has been moved to eleven.
Similarity: 0.6753
Likely paraphrase: False
------------------------------------------------------------
Sentence A: She bought a new laptop.
Sentence B: She purchased a new notebook computer.
Similarity: 0.9237
Likely paraphrase: True
------------------------------------------------------------
Sentence A: How do I change my profile picture?
Sentence B: How does changining my profile picture work?
Similarity: 0.7528
Likely paraphrase: True
------------------------------------------------------------
Sentence A: Th

## 10. Multilingual Semantic Search

A multilingual Sentence Transformer maps texts from different languages into a shared semantic space.

This makes it possible to search documents in one language using a query written in another language.

For multilingual tasks, use a multilingual model rather than an English-only model.


Using the same multilingual model for both queries and documents is crucial because it ensures that all texts, regardless of language, are mapped into the **same shared semantic space**.

This common embedding space allows for direct and meaningful comparison of vectors (e.g., via cosine similarity), enabling accurate cross-lingual semantic search and ensuring that semantically similar texts, even in different languages, are represented as being 'close' to each other.

In [29]:
# Load a multilingual Sentence Transformer model.
multilingual_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [30]:
def multi_semantic_search(multilingual_documents, multilingual_query):
  # Encode the multilingual documents and query with the same model.
  multilingual_document_embeddings = multilingual_model.encode(
      multilingual_documents
  )
  multilingual_query_embedding = multilingual_model.encode(
      [multilingual_query])


  # Calculate cross-language semantic similarity.
  multilingual_scores = cosine_similarity(
      multilingual_query_embedding,
      multilingual_document_embeddings
  )[0]

  # Rank documents according to their similarity to the query.
  multilingual_ranking = np.argsort(multilingual_scores)[::-1]

  # Display the cross-language search results.
  for rank, document_index in enumerate(multilingual_ranking, start=1):
      print(f"{rank}. Score: {multilingual_scores[document_index]:.4f}")
      print(f"   {multilingual_documents[document_index]}")

In [31]:
# Define documents in English, Vietnamese, and Spanish.
multilingual_documents = [
    "Artificial intelligence helps computers solve problems.",
    "Trí tuệ nhân tạo giúp máy tính giải quyết vấn đề.",
    "La inteligencia artificial ayuda a las computadoras a resolver problemas.",
    "A database stores structured information.",
    "Cơ sở dữ liệu lưu trữ thông tin có cấu trúc.",
    "Una base de datos almacena información estructurada."
]

# Define a Vietnamese query.
multilingual_query = "Máy tính có thể hiểu ngôn ngữ của con người như thế nào?"
multi_semantic_search(multilingual_documents, multilingual_query)

1. Score: 0.4175
   La inteligencia artificial ayuda a las computadoras a resolver problemas.
2. Score: 0.4136
   Artificial intelligence helps computers solve problems.
3. Score: 0.4094
   Trí tuệ nhân tạo giúp máy tính giải quyết vấn đề.
4. Score: 0.2515
   A database stores structured information.
5. Score: 0.2145
   Una base de datos almacena información estructurada.
6. Score: 0.2099
   Cơ sở dữ liệu lưu trữ thông tin có cấu trúc.


In [32]:
english_query = "How can a computer understand human language"

multi_semantic_search(multilingual_documents, english_query)

1. Score: 0.4431
   Artificial intelligence helps computers solve problems.
2. Score: 0.4416
   La inteligencia artificial ayuda a las computadoras a resolver problemas.
3. Score: 0.4399
   Trí tuệ nhân tạo giúp máy tính giải quyết vấn đề.
4. Score: 0.2751
   A database stores structured information.
5. Score: 0.2384
   Una base de datos almacena información estructurada.
6. Score: 0.2139
   Cơ sở dữ liệu lưu trữ thông tin có cấu trúc.


In [33]:
# A spanish query
spanish_query = "La inteligencia artificial ayuda a las computadoras a resolver problemas."

multi_semantic_search(multilingual_documents, spanish_query)

1. Score: 1.0000
   La inteligencia artificial ayuda a las computadoras a resolver problemas.
2. Score: 0.9894
   Artificial intelligence helps computers solve problems.
3. Score: 0.9343
   Trí tuệ nhân tạo giúp máy tính giải quyết vấn đề.
4. Score: 0.2825
   A database stores structured information.
5. Score: 0.2603
   Una base de datos almacena información estructurada.
6. Score: 0.2347
   Cơ sở dữ liệu lưu trữ thông tin có cấu trúc.


## 12. Recap

| Application | Main idea | Typical output |
|---|---|---|
| Text embedding | Convert text into a dense vector | Embedding vector |
| Semantic similarity | Compare vector representations | Cosine similarity score |
| Semantic search | Rank documents by meaning | Ranked documents |
| FAQ chatbot | Retrieve the closest known question | Stored answer |
| Text clustering | Group similar embeddings | Cluster assignments |
| Text classification | Use embeddings as model features | Predicted label |
| Recommendation | Find similar item descriptions | Ranked recommendations |
| Paraphrase detection | Compare sentence meanings | Similarity score and decision |
| Multilingual search | Compare texts across languages | Cross-language results |

### Important considerations

- Embeddings capture semantic patterns, but they are not perfect.
- Similarity thresholds should be selected using validation data.
- The quality of the result depends on the embedding model and the data.
- For large collections, use a vector database or an approximate nearest-neighbor index.
- Always evaluate the system using task-specific metrics.
